**Reviewer note:** This notebook is a preserved development experiment. Read `docs/EXPERIMENTS.md` and `docs/METHODOLOGY.md` before execution. Raw/restricted inputs are not included. Historical references to 'test' do not establish prospective independence or validated online causality.

# 0. FEM v2 `common_only`: transfer learning preregistrato

Questo notebook confronta il FEM storico con la variante fisica `common_only`
senza usare le label sintetiche come sostituzione dei target reali.

Vincoli dell'esperimento:

- 668 cross-scan reali, split cronologico 400/133/135;
- il test reale da 135 casi resta bloccato per default;
- FEM v1 e v2 usano le stesse righe e solo 30, 60 e 85 gradi;
- le label v2 vengono collegate tramite `source_row_id` prima di ogni filtro;
- `FEMv2_XEL_arcsec` e' gia' XEL proiettato: non si applica un altro coseno;
- i timestamp FEM duplicati vengono aggregati con la media nel resampling a 2 minuti;
- gli split FEM separano interi gruppi giorno-elevazione;
- seed, inizializzazioni, epoche, ordine delle finestre e fold sono appaiati.

Il notebook salva predizioni riga per riga, metriche globali/per elevazione,
bootstrap appaiato per giornate e decisione del gate. Non modifica gli artefatti
operativi.

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_SEEDS = [17, 29, 43]
FEM_WINDOW_CURVE = [1_000, 3_000, 8_000, 24_000]
COMMON_ELEVATIONS = [30.0, 60.0, 85.0]
SEQ_LEN = 60
N_PCA = 16
BATCH_SIZE = 128
PRETRAIN_EPOCHS = 40
HEAD_EPOCHS = 20
FINETUNE_EPOCHS = 60
SSL_EPOCHS = 40
TEMPORAL_CV_FOLDS = 3
MAX_SSL_WINDOWS = 12_000
SPEC_ARCSEC = 5.0
MATERIAL_ABSOLUTE_ARCSEC = 0.30
MATERIAL_RELATIVE = 0.03
MAX_SINGLE_FOLD_REGRESSION = 0.25

RUN_VALIDATION_CURVE = True
RUN_TEMPORAL_CV = True
RUN_LOCKED_TEST = False
LOCKED_TEST_CONFIRMATION = ""  

tf.keras.utils.set_random_seed(RANDOM_SEEDS[0])
sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/metrologyml")]
    for candidate in candidates:
        if (candidate / "src" / "aims_pipeline.py").exists():
            return candidate.resolve()
    raise FileNotFoundError('ML_metrologia project directory not found')


ROOT = find_project_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from fem_v2_labels import attach_fem_v2_labels
from aims_pipeline import ThermalFeatureBuilder, prepare_samples, receiver_one_hot
from aims_training import el_model, xel_model

REAL_PATH = ROOT / "data" / "raw" / "real" / "EffectiveDataset_Tseries_with_azimuth.csv"
TARGET_AUDIT_PATH = ROOT / "data" / "raw" / "control" / "control_crosscheck_effective_target_projection.csv"
POINTING_AUDIT_PATH = ROOT / "data" / "raw" / "control" / "control_crosscheck_pointing_audit.csv"
FEM_SOURCE_PATH = ROOT / "data" / "raw" / "fem" / "Dataset (1).xlsx"
FEM_V2_LABELS_PATH = ROOT / "data" / "processed" / "fem_v2_common_only_labels_30_60_85.csv"
FEM_V2_MANIFEST_PATH = ROOT / "data" / "processed" / "fem_v2_common_only_labels_30_60_85.manifest.json"
OUTPUT_DIR = ROOT / "outputs" / "experiments" / "fem_v2_common_only"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

samples = prepare_samples(REAL_PATH, TARGET_AUDIT_PATH, POINTING_AUDIT_PATH)
X_real = samples.sequences.astype("float32")
meta = samples.metadata.reset_index(drop=True).copy()
meta["Timestamp"] = pd.to_datetime(meta["Timestamp"])
y_real = {
    "XEL": samples.target_xel.astype("float32"),
    "EL": samples.target_el.astype("float32"),
}
train_idx = np.flatnonzero(meta["Split"].eq("train"))
val_idx = np.flatnonzero(meta["Split"].eq("validation"))
test_idx = np.flatnonzero(meta["Split"].eq("test"))
if (len(meta), len(train_idx), len(val_idx), len(test_idx)) != (668, 400, 133, 135):
    raise ValueError('Unexpected real-data split; the locked test cannot be redefined')

real_data_summary_df = pd.DataFrame([{
    "samples": len(meta), "train": len(train_idx), "validation": len(val_idx),
    "locked_test": len(test_idx), "sequence_shape": str(tuple(X_real.shape[1:])),
    "validation_start": meta.loc[val_idx, "Timestamp"].min(),
    "test_start": meta.loc[test_idx, "Timestamp"].min(),
    "locked_test_used": False,
}])
display(real_data_summary_df)

In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


manifest = json.loads(FEM_V2_MANIFEST_PATH.read_text(encoding="utf-8"))
source_hash = sha256_file(FEM_SOURCE_PATH)
labels_hash = sha256_file(FEM_V2_LABELS_PATH)
if source_hash.lower() != manifest["source_workbook_sha256"].lower():
    raise ValueError('FEM workbook hash differs from the manifest')
if labels_hash.lower() != manifest["labels_sha256"].lower():
    raise ValueError('FEM-v2 label hash differs from the manifest')

print('Loading the 356,400-row FEM workbook...')
fem_raw = pd.read_excel(FEM_SOURCE_PATH, engine="openpyxl")
source_rows = len(fem_raw)
rows_in_duplicate_groups_raw = int(
    fem_raw.duplicated(["Time", "Elevation"], keep=False).sum()
)
duplicate_excess_raw = int(
    fem_raw.duplicated(["Time", "Elevation"], keep="first").sum()
)
fem_common = attach_fem_v2_labels(fem_raw, FEM_V2_LABELS_PATH)
timestamp_alignment_max_abs_ms = fem_common.attrs.get(
    "timestamp_alignment_max_abs_ms", np.nan
)
fem_common["Time"] = pd.to_datetime(fem_common["Time"], format="mixed", errors="raise")
fem_common["Elevation"] = pd.to_numeric(fem_common["Elevation"], errors="raise")
if sorted(fem_common["Elevation"].unique().astype(float)) != COMMON_ELEVATIONS:
    raise ValueError('Common elevations must be 30, 60 and 85 degrees')

sensor_cols = sorted(
    [column for column in fem_common.columns if re.match(r"^Sensors_\s*\d+$", str(column))],
    key=lambda column: int(re.search(r"(\d+)$", str(column)).group(1)),
)
if len(sensor_cols) != 295:
    raise ValueError(f"Expected 295 FEM sensors, found {len(sensor_cols)}")
fem_common = fem_common.rename(columns={
    column: "Sensors_{:03d}".format(
        int(re.search(r"(\d+)$", str(column)).group(1))
    )
    for column in sensor_cols
})
fem_sensor_cols = sorted(
    [column for column in fem_common.columns if re.match(r"^Sensors_\d{3}$", column)],
    key=lambda column: int(column.split("_")[1]),
)
required_numeric = fem_sensor_cols + [
    "FEMv1_XEL_arcsec", "FEMv1_EL_arcsec",
    "FEMv2_XEL_arcsec", "FEMv2_EL_arcsec",
]
for column in required_numeric:
    fem_common[column] = pd.to_numeric(fem_common[column], errors="coerce")
fem_common = fem_common.dropna(subset=["Time", "Elevation", *required_numeric]).copy()

fem_join_audit_df = pd.DataFrame([{
    "source_rows": source_rows, "matched_common_rows": len(fem_common),
    "sensor_count": len(fem_sensor_cols),
    "rows_in_duplicate_groups_raw": rows_in_duplicate_groups_raw,
    "duplicate_excess_rows_raw": duplicate_excess_raw,
    "timestamp_alignment_max_abs_ms": timestamp_alignment_max_abs_ms,
    "source_sha256": source_hash, "labels_sha256": labels_hash,
    "elevations": ",".join(map(lambda value: str(int(value)), COMMON_ELEVATIONS)),
    "v2_XEL_cosine_applied": False,
}])
display(fem_join_audit_df)
display(
    fem_common.groupby("Elevation", as_index=False)
    .agg(
        rows=("source_row_id", "size"),
        v1_XEL_std=("FEMv1_XEL_arcsec", "std"),
        v2_XEL_std=("FEMv2_XEL_arcsec", "std"),
        v1_EL_std=("FEMv1_EL_arcsec", "std"),
        v2_EL_std=("FEMv2_EL_arcsec", "std"),
    )
)
del fem_raw
gc.collect()

In [ ]:
class RealDomainThermalProjector:
    def __init__(self, n_components=N_PCA):
        self.mean_scaler = StandardScaler()
        self.pattern_scaler = StandardScaler()
        self.pattern_pca = PCA(n_components=n_components, random_state=RANDOM_SEEDS[0])

    def fit(self, sequences):
        frames = np.asarray(sequences, dtype="float32").reshape(-1, sequences.shape[-1])
        means = frames.mean(axis=1, keepdims=True)
        centered = frames - means
        self.mean_scaler.fit(means)
        self.pattern_pca.fit(self.pattern_scaler.fit_transform(centered))
        return self

    def transform_frames(self, frames):
        frames = np.asarray(frames, dtype="float32")
        means = frames.mean(axis=1, keepdims=True)
        centered = frames - means
        return np.concatenate([
            self.mean_scaler.transform(means),
            self.pattern_pca.transform(self.pattern_scaler.transform(centered)),
        ], axis=1).astype("float32")

    def transform(self, sequences):
        shape = sequences.shape
        values = self.transform_frames(sequences.reshape(-1, shape[-1]))
        return values.reshape(shape[0], shape[1], -1)


RECEIVER_LEVELS = ["K", "C_high", "C_low", "Q", "unknown"]


def context_features(elevation, receiver_band=None):
    elev = np.asarray(elevation, dtype="float32").reshape(-1)
    if receiver_band is None:
        receiver_band = np.repeat("unknown", len(elev))
    category = pd.Categorical(receiver_band, categories=RECEIVER_LEVELS)
    one_hot = pd.get_dummies(category, dtype=float).to_numpy(dtype="float32")
    return np.column_stack([
        elev / 90.0, np.sin(np.deg2rad(elev)), np.cos(np.deg2rad(elev)), one_hot,
    ]).astype("float32")


projector = RealDomainThermalProjector().fit(X_real[train_idx])
Z_real = projector.transform(X_real)
C_real = context_features(meta["Elevation"], meta["ReceiverBand"].fillna("unknown"))
representation_summary_df = pd.DataFrame([{
    "fit_domain": "real_train_only", "train_sequences": len(train_idx),
    "PCA_components": N_PCA, "sequence_features": Z_real.shape[-1],
    "context_features": C_real.shape[-1],
    "pattern_variance_explained_percent": 100 * projector.pattern_pca.explained_variance_ratio_.sum(),
    "locked_test_used_for_fit": False,
}])
display(representation_summary_df)

In [ ]:
LABEL_COLUMNS = [
    "FEMv1_XEL_arcsec", "FEMv1_EL_arcsec",
    "FEMv2_XEL_arcsec", "FEMv2_EL_arcsec",
]


def build_fem_group_bank(frame, sensors):
    groups, candidates = {}, []
    rows_in_duplicate_groups = 0
    duplicate_excess_rows = 0
    work = frame.copy()
    work["day"] = work["Time"].dt.floor("D")
    for (elevation, day), group in work.groupby(["Elevation", "day"], sort=True):
        group = group.sort_values(["Time", "source_row_id"])
        rows_in_duplicate_groups += int(group["Time"].duplicated(keep=False).sum())
        duplicate_excess_rows += int(group["Time"].duplicated(keep="first").sum())
        values = group.set_index("Time")
        averaged = values[sensors + LABEL_COLUMNS].resample("2min").mean()
        source_ids = values[["source_row_id"]].resample("2min").max()
        sampled = pd.concat([averaged, source_ids], axis=1).dropna()
        if len(sampled) < SEQ_LEN:
            continue
        group_id = f"{pd.Timestamp(day).date()}__{int(round(elevation))}"
        groups[group_id] = {
            "sensors": sampled[sensors].to_numpy(dtype="float32"),
            "labels": sampled[LABEL_COLUMNS].to_numpy(dtype="float32"),
            "times": sampled.index.to_numpy(),
            "source_ids": sampled["source_row_id"].round().astype("int64").to_numpy(),
            "elevation": float(elevation), "day": pd.Timestamp(day),
        }
        for endpoint in range(SEQ_LEN - 1, len(sampled)):
            candidates.append({
                "candidate_id": len(candidates), "group_id": group_id,
                "day": pd.Timestamp(day), "elevation": float(elevation),
                "endpoint": endpoint, "end_timestamp": sampled.index[endpoint],
                "end_source_row_id": int(sampled["source_row_id"].iloc[endpoint]),
            })
    return (
        groups, pd.DataFrame(candidates),
        rows_in_duplicate_groups, duplicate_excess_rows,
    )


def assign_group_splits(candidates):
    group_table = candidates[["group_id", "day", "elevation"]].drop_duplicates()
    assignments = []
    for _, subset in group_table.groupby("elevation", sort=True):
        subset = subset.sort_values(["day", "group_id"]).reset_index(drop=True)
        cut = max(1, int(np.floor(0.90 * len(subset))))
        subset["fem_split"] = np.where(subset.index < cut, "train", "validation")
        assignments.append(subset)
    assignment = pd.concat(assignments, ignore_index=True)
    result = candidates.merge(
        assignment[["group_id", "fem_split"]], on="group_id", validate="many_to_one"
    )
    overlap = set(result.query("fem_split == 'train'")["group_id"]) & set(
        result.query("fem_split == 'validation'")["group_id"]
    )
    if overlap:
        raise RuntimeError('FEM group leakage detected')
    return result


(
    fem_groups, fem_candidates,
    rows_in_duplicate_groups_common, duplicate_excess_common,
) = build_fem_group_bank(fem_common, fem_sensor_cols)
fem_candidates = assign_group_splits(fem_candidates)
fixed_rng = np.random.default_rng(20260921)
train_candidate_ids = fem_candidates.query("fem_split == 'train'")[
    "candidate_id"
].to_numpy(dtype=int)
fixed_rng.shuffle(train_candidate_ids)
validation_candidate_ids = fem_candidates.query("fem_split == 'validation'")[
    "candidate_id"
].to_numpy(dtype=int)
if len(train_candidate_ids) < max(FEM_WINDOW_CURVE):
    raise ValueError('Insufficient FEM windows for the preregistered learning curve')


def materialize_fem(candidate_ids, fitted_projector):
    selected = fem_candidates.set_index("candidate_id").loc[np.asarray(candidate_ids)]
    selected_groups = set(selected["group_id"])
    transformed = {
        group_id: fitted_projector.transform_frames(group["sensors"])
        for group_id, group in fem_groups.items() if group_id in selected_groups
    }
    x_out, c_out, y1_out, y2_out, key_rows = [], [], [], [], []
    for candidate_id, row in selected.iterrows():
        group = fem_groups[row["group_id"]]
        endpoint = int(row["endpoint"])
        x_out.append(transformed[row["group_id"]][endpoint - SEQ_LEN + 1:endpoint + 1])
        c_out.append(context_features([row["elevation"]])[0])
        labels = group["labels"][endpoint]
        y1_out.append(labels[:2])
        y2_out.append(labels[2:])
        key_rows.append({
            "candidate_id": int(candidate_id), "group_id": row["group_id"],
            "day": row["day"], "elevation": row["elevation"],
            "end_timestamp": row["end_timestamp"],
            "end_source_row_id": row["end_source_row_id"],
        })
    return {
        "Z": np.asarray(x_out, dtype="float32"),
        "C": np.asarray(c_out, dtype="float32"),
        "Y_v1": np.asarray(y1_out, dtype="float32"),
        "Y_v2": np.asarray(y2_out, dtype="float32"),
        "keys": pd.DataFrame(key_rows),
    }


fem_train_bank = materialize_fem(train_candidate_ids[:max(FEM_WINDOW_CURVE)], projector)
fem_validation_bank = materialize_fem(validation_candidate_ids, projector)
if not fem_train_bank["keys"]["candidate_id"].is_unique:
    raise RuntimeError('FEM window keys are not unique')
fem_window_summary_df = pd.DataFrame([{
    "common_rows": len(fem_common), "groups": len(fem_groups),
    "train_groups": fem_candidates.query("fem_split == 'train'")["group_id"].nunique(),
    "validation_groups": fem_candidates.query("fem_split == 'validation'")["group_id"].nunique(),
    "candidate_windows": len(fem_candidates),
    "fixed_train_windows": len(fem_train_bank["Z"]),
    "validation_windows": len(fem_validation_bank["Z"]),
    "rows_in_duplicate_groups_common_elevations": rows_in_duplicate_groups_common,
    "duplicate_excess_rows_common_elevations": duplicate_excess_common,
    "sequence_shape": str(tuple(fem_train_bank["Z"].shape[1:])),
}])
display(fem_window_summary_df)
display(fem_train_bank["keys"].head())
del fem_common
gc.collect()

In [ ]:
def normalized_sensor_name(name):
    match = re.match(r"^Sensors_\s*(\d+)$", str(name).strip())
    return f"Sensors_{int(match.group(1)):03d}" if match else str(name).strip()


def valid_endpoints(times, seq_len=SEQ_LEN, max_gap_seconds=180):
    times = pd.to_datetime(times).to_numpy(dtype="datetime64[ns]")
    delta = np.diff(times).astype("timedelta64[s]").astype(float)
    bad = (delta > max_gap_seconds) | (delta <= 0)
    prefix = np.r_[0, np.cumsum(bad.astype(int))]
    return np.asarray([
        end for end in range(seq_len - 1, len(times))
        if prefix[end] - prefix[end - seq_len + 1] == 0
    ], dtype=int)


def load_ssl_sequences(fitted_projector, cutoff, seed, max_windows=MAX_SSL_WINDOWS):
    raw = pd.read_csv(REAL_PATH, low_memory=False).rename(columns=normalized_sensor_name)
    raw["Timestamp"] = pd.to_datetime(raw["Timestamp"], errors="coerce")
    sensors = sorted(
        [column for column in raw.columns if re.match(r"^Sensors_\d{3}$", column)],
        key=lambda column: int(column.split("_")[1]),
    )
    before = raw.loc[raw["Timestamp"].lt(cutoff), ["Timestamp", *sensors]].dropna()
    before["GridTime"] = before["Timestamp"].dt.floor("2min")
    grid = before.groupby("GridTime", sort=True)[sensors].mean().ffill(limit=3).dropna()
    endpoints = valid_endpoints(grid.index)
    rng = np.random.default_rng(seed)
    if len(endpoints) > max_windows:
        endpoints = np.sort(rng.choice(endpoints, max_windows, replace=False))
    frames = fitted_projector.transform_frames(grid.to_numpy(dtype="float32"))
    sequences = np.stack([
        frames[end - SEQ_LEN + 1:end + 1] for end in endpoints
    ]).astype("float32")
    return sequences, {
        "telemetry_rows_before_cutoff": len(before), "ssl_windows": len(sequences),
        "cutoff": cutoff, "first_time": grid.index.min(), "last_time": grid.index.max(),
        "future_rows_used": False,
    }


Z_ssl, ssl_summary = load_ssl_sequences(
    projector, meta.loc[val_idx, "Timestamp"].min(), RANDOM_SEEDS[0]
)
ssl_data_summary_df = pd.DataFrame([ssl_summary])
display(ssl_data_summary_df)

In [ ]:
AXIS_INDEX = {"XEL": 0, "EL": 1}


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    residual = y_true - y_pred
    absolute = np.abs(residual)
    return {
        "count": len(y_true), "MAE": float(absolute.mean()),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)), "bias": float(residual.mean()),
        "within_5_percent": float(100 * np.mean(absolute <= SPEC_ARCSEC)),
        "p90_abs_error": float(np.quantile(absolute, 0.90)),
    }


def make_regressor(input_dim, context_dim):
    sequence = keras.Input((SEQ_LEN, input_dim), name="sequence")
    context = keras.Input((context_dim,), name="context")
    encoded = layers.GRU(32, dropout=0.10, name="thermal_gru")(sequence)
    encoded = layers.Dense(16, activation="gelu", name="thermal_projection")(encoded)
    ctx = layers.Dense(12, activation="gelu", name="context_projection")(context)
    fused = layers.Concatenate(name="fusion")([encoded, ctx])
    hidden = layers.Dense(20, activation="gelu", name="supervised_hidden")(fused)
    hidden = layers.Dropout(0.20, name="supervised_dropout")(hidden)
    output = layers.Dense(1, name="supervised_output")(hidden)
    return keras.Model([sequence, context], output)


def layer_state(model):
    return {layer.name: layer.get_weights() for layer in model.layers if layer.get_weights()}


def restore_state(model, state, excluded=()):
    for layer in model.layers:
        if layer.name in state and layer.name not in excluded and layer.get_weights():
            layer.set_weights(state[layer.name])
    return model


def callbacks(patience=8):
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", patience=max(3, patience // 2), factor=0.5,
            min_lr=1e-6, verbose=0,
        ),
    ]


def run_ssl_pretraining(Z, seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    sequence = keras.Input((SEQ_LEN, Z.shape[-1]))
    encoded = layers.GRU(32, dropout=0.10, name="thermal_gru")(sequence)
    encoded = layers.Dense(16, activation="gelu", name="thermal_projection")(encoded)
    repeated = layers.RepeatVector(SEQ_LEN)(encoded)
    decoded = layers.GRU(32, return_sequences=True)(repeated)
    output = layers.TimeDistributed(layers.Dense(Z.shape[-1]))(decoded)
    model = keras.Model(sequence, output)
    rng = np.random.default_rng(seed)
    corrupted = Z.copy()
    mask = rng.random(corrupted.shape[:2]) < 0.15
    corrupted[mask] = 0.0
    corrupted += rng.normal(0, 0.02, corrupted.shape).astype("float32")
    cut = max(1, int(0.90 * len(Z)))
    model.compile(keras.optimizers.Adam(3e-4, clipnorm=1.0), loss="mse")
    model.fit(
        corrupted[:cut], Z[:cut], validation_data=(corrupted[cut:], Z[cut:]),
        epochs=SSL_EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks(), verbose=0,
    )
    return {
        name: model.get_layer(name).get_weights()
        for name in ["thermal_gru", "thermal_projection"]
    }


def run_fem_pretraining(bank_train, bank_val, axis, label_version, sample_count, seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    axis_index = AXIS_INDEX[axis]
    y_key = "Y_v1" if label_version == "v1" else "Y_v2"
    y_train = bank_train[y_key][:sample_count, axis_index]
    y_val = bank_val[y_key][:, axis_index]
    scaler = StandardScaler().fit(y_train.reshape(-1, 1))
    model = make_regressor(bank_train["Z"].shape[-1], bank_train["C"].shape[-1])
    model.compile(keras.optimizers.Adam(3e-4, clipnorm=1.0), loss=keras.losses.Huber())
    model.fit(
        [bank_train["Z"][:sample_count], bank_train["C"][:sample_count]],
        scaler.transform(y_train.reshape(-1, 1)),
        validation_data=(
            [bank_val["Z"], bank_val["C"]], scaler.transform(y_val.reshape(-1, 1))
        ),
        epochs=PRETRAIN_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks(), verbose=0,
    )
    return layer_state(model)


def fit_real_regressor(
    Z_train, C_train, y_train, Z_eval, C_eval, y_eval, seed,
    pretrained_state=None, ssl_state=None, reset_supervised_head=False,
):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    scaler = StandardScaler().fit(np.asarray(y_train).reshape(-1, 1))
    model = make_regressor(Z_train.shape[-1], C_train.shape[-1])
    if pretrained_state is not None:
        excluded = (
            "context_projection", "supervised_hidden", "supervised_output"
        ) if reset_supervised_head else ()
        restore_state(model, pretrained_state, excluded=excluded)
    if ssl_state is not None:
        restore_state(model, ssl_state)
    if pretrained_state is not None or ssl_state is not None:
        model.get_layer("thermal_gru").trainable = False
        model.get_layer("thermal_projection").trainable = False
        model.compile(keras.optimizers.Adam(5e-4, clipnorm=1.0), loss=keras.losses.Huber())
        model.fit(
            [Z_train, C_train], scaler.transform(np.asarray(y_train).reshape(-1, 1)),
            validation_data=(
                [Z_eval, C_eval], scaler.transform(np.asarray(y_eval).reshape(-1, 1))
            ), epochs=HEAD_EPOCHS, batch_size=64, callbacks=callbacks(6), verbose=0,
        )
        model.get_layer("thermal_gru").trainable = True
        model.get_layer("thermal_projection").trainable = True
    model.compile(keras.optimizers.Adam(1e-4, clipnorm=1.0), loss=keras.losses.Huber())
    model.fit(
        [Z_train, C_train], scaler.transform(np.asarray(y_train).reshape(-1, 1)),
        validation_data=(
            [Z_eval, C_eval], scaler.transform(np.asarray(y_eval).reshape(-1, 1))
        ), epochs=FINETUNE_EPOCHS, batch_size=64, callbacks=callbacks(10), verbose=0,
    )
    prediction = scaler.inverse_transform(
        model.predict([Z_eval, C_eval], verbose=0)
    ).reshape(-1)
    return prediction, layer_state(model), scaler


def production_prediction(axis, fit_idx, eval_idx):
    builder = ThermalFeatureBuilder().fit(X_real[fit_idx])
    fit_meta = meta.iloc[fit_idx].reset_index(drop=True)
    eval_meta = meta.iloc[eval_idx].reset_index(drop=True)
    fit_features = builder.transform(
        X_real[fit_idx], samples.elevation[fit_idx], fit_meta
    )
    fit_features = pd.concat([
        fit_features, receiver_one_hot(fit_meta)
    ], axis=1)
    eval_features = builder.transform(
        X_real[eval_idx], samples.elevation[eval_idx], eval_meta
    )
    eval_features = pd.concat([
        eval_features, receiver_one_hot(eval_meta)
    ], axis=1)
    model = (xel_model() if axis == "XEL" else el_model()).fit(
        fit_features, y_real[axis][fit_idx]
    )
    return model.predict(eval_features)

In [ ]:
def day_block_bootstrap(y_true, candidate, reference, timestamps, n_boot=5000, seed=31):
    table = pd.DataFrame({
        "y": np.asarray(y_true), "candidate": np.asarray(candidate),
        "reference": np.asarray(reference), "time": pd.to_datetime(timestamps),
    })
    table["day"] = table["time"].dt.floor("D")
    blocks = [np.asarray(index, dtype=int) for index in table.groupby("day").groups.values()]
    rng = np.random.default_rng(seed)
    deltas = []
    for _ in range(n_boot):
        chosen = rng.integers(0, len(blocks), size=len(blocks))
        idx = np.concatenate([blocks[value] for value in chosen])
        candidate_mae = np.abs(table.loc[idx, "y"] - table.loc[idx, "candidate"]).mean()
        reference_mae = np.abs(table.loc[idx, "y"] - table.loc[idx, "reference"]).mean()
        deltas.append(candidate_mae - reference_mae)
    values = np.asarray(deltas)
    return {
        "delta_MAE": float(values.mean()),
        "ci95_low": float(np.quantile(values, 0.025)),
        "ci95_high": float(np.quantile(values, 0.975)),
        "probability_better_percent": float(100 * np.mean(values < 0)),
    }


def append_predictions(rows, stage, axis, variant, seed, fold, fem_windows, indices, prediction):
    for row_index, value in zip(indices, prediction):
        rows.append({
            "stage": stage, "axis": axis, "variant": variant, "seed": seed,
            "fold": fold, "fem_windows": fem_windows, "row_index": int(row_index),
            "Timestamp": meta.loc[row_index, "Timestamp"],
            "Elevation": float(meta.loc[row_index, "Elevation"]),
            "target": float(y_real[axis][row_index]), "prediction": float(value),
        })


def metrics_from_prediction_rows(frame):
    rows = []
    grouping = ["stage", "axis", "variant", "seed", "fold", "fem_windows"]
    for keys, subset in frame.groupby(grouping, dropna=False, sort=False):
        rows.append(dict(zip(grouping, keys)) | regression_metrics(
            subset["target"], subset["prediction"]
        ))
    return pd.DataFrame(rows)

In [ ]:
validation_prediction_rows = []
validation_curve_metrics_df = pd.DataFrame()
if RUN_VALIDATION_CURVE:
    for axis in ["XEL", "EL"]:
        production = production_prediction(axis, train_idx, val_idx)
        append_predictions(
            validation_prediction_rows, "validation_curve", axis,
            "production_final", -1, 0, 0, val_idx, production,
        )
        for seed in RANDOM_SEEDS:
            ssl_state = run_ssl_pretraining(Z_ssl, seed)
            random_prediction, _, _ = fit_real_regressor(
                Z_real[train_idx], C_real[train_idx], y_real[axis][train_idx],
                Z_real[val_idx], C_real[val_idx], y_real[axis][val_idx], seed,
            )
            ssl_prediction, _, _ = fit_real_regressor(
                Z_real[train_idx], C_real[train_idx], y_real[axis][train_idx],
                Z_real[val_idx], C_real[val_idx], y_real[axis][val_idx], seed,
                ssl_state=ssl_state,
            )
            append_predictions(
                validation_prediction_rows, "validation_curve", axis,
                "real_random_init", seed, 0, 0, val_idx, random_prediction,
            )
            append_predictions(
                validation_prediction_rows, "validation_curve", axis,
                "real_SSL", seed, 0, 0, val_idx, ssl_prediction,
            )
            for window_count in FEM_WINDOW_CURVE:
                v1_state = run_fem_pretraining(
                    fem_train_bank, fem_validation_bank, axis, "v1", window_count, seed
                )
                v2_state = run_fem_pretraining(
                    fem_train_bank, fem_validation_bank, axis, "v2", window_count, seed
                )
                for variant, state, reset_head in [
                    ("FEM_v1_pretrain", v1_state, False),
                    ("FEM_v2_pretrain", v2_state, False),
                    ("FEM_v2_reset_head", v2_state, True),
                ]:
                    prediction, _, _ = fit_real_regressor(
                        Z_real[train_idx], C_real[train_idx], y_real[axis][train_idx],
                        Z_real[val_idx], C_real[val_idx], y_real[axis][val_idx], seed,
                        pretrained_state=state, reset_supervised_head=reset_head,
                    )
                    append_predictions(
                        validation_prediction_rows, "validation_curve", axis,
                        variant, seed, 0, window_count, val_idx, prediction,
                    )
                gc.collect()

    validation_predictions_df = pd.DataFrame(validation_prediction_rows)
    validation_curve_metrics_df = metrics_from_prediction_rows(validation_predictions_df)
    display(validation_curve_metrics_df.sort_values(["axis", "fem_windows", "MAE"]))
else:
    validation_predictions_df = pd.DataFrame()
    print("RUN_VALIDATION_CURVE=False")

In [ ]:
validation_curve_summary_df = pd.DataFrame()
validation_v2_vs_v1_bootstrap_df = pd.DataFrame()
if len(validation_curve_metrics_df):
    validation_curve_summary_df = (
        validation_curve_metrics_df.groupby(
            ["axis", "variant", "fem_windows"], as_index=False
        ).agg(
            seeds=("seed", "nunique"), MAE_mean=("MAE", "mean"),
            MAE_std=("MAE", "std"), bias_mean=("bias", "mean"),
            p90_mean=("p90_abs_error", "mean"),
            within_5_mean=("within_5_percent", "mean"),
        ).sort_values(["axis", "fem_windows", "MAE_mean"])
    )
    bootstrap_rows = []
    for axis in ["XEL", "EL"]:
        for window_count in FEM_WINDOW_CURVE:
            subset = validation_predictions_df.query(
                "axis == @axis and fem_windows == @window_count"
            )
            wide = subset.pivot_table(
                index=["row_index", "Timestamp", "target"],
                columns="variant", values="prediction", aggfunc="mean",
            ).reset_index()
            for candidate in ["FEM_v2_pretrain", "FEM_v2_reset_head"]:
                comparison = wide.dropna(subset=[candidate, "FEM_v1_pretrain"])
                bootstrap_rows.append({
                    "axis": axis, "fem_windows": window_count,
                    "candidate": candidate, "reference": "FEM_v1_pretrain",
                    **day_block_bootstrap(
                        comparison["target"], comparison[candidate],
                        comparison["FEM_v1_pretrain"], comparison["Timestamp"],
                    ),
                })
    validation_v2_vs_v1_bootstrap_df = pd.DataFrame(bootstrap_rows)
    display(validation_curve_summary_df)
    display(validation_v2_vs_v1_bootstrap_df)

In [ ]:
def temporal_fold_bounds(indices, n_folds=TEMPORAL_CV_FOLDS):
    indices = np.asarray(indices)
    initial = max(180, len(indices) // 2)
    remaining = len(indices) - initial
    step = max(1, remaining // n_folds)
    folds = []
    for fold in range(n_folds):
        train_end = initial + fold * step
        eval_end = len(indices) if fold == n_folds - 1 else min(len(indices), train_end + step)
        if eval_end > train_end:
            folds.append((indices[:train_end], indices[train_end:eval_end]))
    return folds


cv_prediction_rows = []
cv_metrics_df = pd.DataFrame()
if RUN_TEMPORAL_CV:
    dev_idx = np.flatnonzero(~meta["Split"].eq("test"))
    for fold, (fold_train, fold_eval) in enumerate(temporal_fold_bounds(dev_idx), 1):
        fold_projector = RealDomainThermalProjector().fit(X_real[fold_train])
        fold_Z_real = fold_projector.transform(X_real)
        fold_fem_train = materialize_fem(
            train_candidate_ids[:max(FEM_WINDOW_CURVE)], fold_projector
        )
        fold_fem_validation = materialize_fem(validation_candidate_ids, fold_projector)
        fold_Z_ssl, _ = load_ssl_sequences(
            fold_projector, meta.loc[fold_eval, "Timestamp"].min(),
            RANDOM_SEEDS[0] + fold,
        )
        for axis in ["XEL", "EL"]:
            production = production_prediction(axis, fold_train, fold_eval)
            append_predictions(
                cv_prediction_rows, "temporal_cv", axis, "production_final",
                -1, fold, 0, fold_eval, production,
            )
            for seed in RANDOM_SEEDS:
                ssl_state = run_ssl_pretraining(fold_Z_ssl, seed)
                v1_state = run_fem_pretraining(
                    fold_fem_train, fold_fem_validation, axis, "v1",
                    max(FEM_WINDOW_CURVE), seed,
                )
                v2_state = run_fem_pretraining(
                    fold_fem_train, fold_fem_validation, axis, "v2",
                    max(FEM_WINDOW_CURVE), seed,
                )
                configurations = [
                    ("real_random_init", None, None, False),
                    ("real_SSL", None, ssl_state, False),
                    ("FEM_v1_pretrain", v1_state, None, False),
                    ("FEM_v2_pretrain", v2_state, None, False),
                    ("FEM_v2_reset_head", v2_state, None, True),
                ]
                for variant, pretrained, ssl, reset_head in configurations:
                    prediction, _, _ = fit_real_regressor(
                        fold_Z_real[fold_train], C_real[fold_train], y_real[axis][fold_train],
                        fold_Z_real[fold_eval], C_real[fold_eval], y_real[axis][fold_eval], seed,
                        pretrained_state=pretrained, ssl_state=ssl,
                        reset_supervised_head=reset_head,
                    )
                    append_predictions(
                        cv_prediction_rows, "temporal_cv", axis, variant, seed,
                        fold, max(FEM_WINDOW_CURVE) if variant.startswith("FEM") else 0,
                        fold_eval, prediction,
                    )
        gc.collect()
    cv_predictions_df = pd.DataFrame(cv_prediction_rows)
    cv_metrics_df = metrics_from_prediction_rows(cv_predictions_df)
    display(cv_metrics_df.sort_values(["axis", "fold", "MAE"]))
else:
    cv_predictions_df = pd.DataFrame()
    print("RUN_TEMPORAL_CV=False")

In [ ]:
cv_summary_df = pd.DataFrame()
cv_bootstrap_df = pd.DataFrame()
cv_by_elevation_df = pd.DataFrame()
fem_v2_gate_df = pd.DataFrame()
if len(cv_metrics_df):
    cv_summary_df = (
        cv_metrics_df.groupby(["axis", "variant"], as_index=False)
        .agg(
            folds=("fold", "nunique"), seeds=("seed", "nunique"),
            MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
            bias_mean=("bias", "mean"), p90_mean=("p90_abs_error", "mean"),
            within_5_mean=("within_5_percent", "mean"),
        ).sort_values(["axis", "MAE_mean"])
    )
    ensemble = (
        cv_predictions_df.groupby(
            ["axis", "variant", "row_index", "Timestamp", "Elevation", "target"],
            as_index=False,
        )["prediction"].mean()
    )
    ensemble["elevation_band"] = pd.cut(
        ensemble["Elevation"], [0, 20, 30, 45, 60, 75, 90], include_lowest=True
    )
    elevation_rows = []
    for keys, subset in ensemble.groupby(
        ["axis", "variant", "elevation_band"], observed=True
    ):
        elevation_rows.append({
            "axis": keys[0], "variant": keys[1], "elevation_band": keys[2],
            **regression_metrics(subset["target"], subset["prediction"]),
        })
    cv_by_elevation_df = pd.DataFrame(elevation_rows)

    bootstrap_rows = []
    for axis in ["XEL", "EL"]:
        wide = ensemble.loc[ensemble["axis"].eq(axis)].pivot_table(
            index=["row_index", "Timestamp", "target"],
            columns="variant", values="prediction", aggfunc="first",
        ).reset_index()
        for candidate, reference in [
            ("FEM_v2_pretrain", "FEM_v1_pretrain"),
            ("FEM_v2_reset_head", "FEM_v1_pretrain"),
            ("FEM_v2_pretrain", "real_random_init"),
            ("FEM_v2_reset_head", "real_random_init"),
            ("FEM_v2_pretrain", "real_SSL"),
            ("FEM_v2_reset_head", "real_SSL"),
            ("FEM_v2_pretrain", "production_final"),
            ("FEM_v2_reset_head", "production_final"),
        ]:
            common = wide.dropna(subset=[candidate, reference])
            bootstrap_rows.append({
                "axis": axis, "candidate": candidate, "reference": reference,
                **day_block_bootstrap(
                    common["target"], common[candidate], common[reference], common["Timestamp"]
                ),
            })
    cv_bootstrap_df = pd.DataFrame(bootstrap_rows)

    gate_rows = []
    for axis in ["XEL", "EL"]:
        summary = cv_summary_df.loc[cv_summary_df["axis"].eq(axis)].set_index("variant")
        for priority, candidate in enumerate(
            ["FEM_v2_reset_head", "FEM_v2_pretrain"], start=1
        ):
            reference = "FEM_v1_pretrain"
            v1_mae = float(summary.loc[reference, "MAE_mean"])
            v2_mae = float(summary.loc[candidate, "MAE_mean"])
            threshold = max(MATERIAL_ABSOLUTE_ARCSEC, MATERIAL_RELATIVE * v1_mae)
            paired = cv_bootstrap_df.loc[
                cv_bootstrap_df["axis"].eq(axis)
                & cv_bootstrap_df["candidate"].eq(candidate)
                & cv_bootstrap_df["reference"].eq(reference)
            ].iloc[0]
            fold_comparison = cv_metrics_df.loc[
                cv_metrics_df["axis"].eq(axis)
                & cv_metrics_df["variant"].isin([candidate, reference])
            ].groupby(["fold", "variant"])["MAE"].mean().unstack()
            worst_fold_delta = float(
                (fold_comparison[candidate] - fold_comparison[reference]).max()
            )
            curve = validation_curve_metrics_df.loc[
                validation_curve_metrics_df["axis"].eq(axis)
                & validation_curve_metrics_df["variant"].isin([candidate, reference])
                & validation_curve_metrics_df["fem_windows"].isin([8_000, 24_000])
            ]
            curve_mean = curve.groupby(["fem_windows", "variant"])["MAE"].mean().unstack()
            persistent_8k_24k = bool(
                len(curve_mean) == 2
                and (curve_mean[candidate] < curve_mean[reference]).all()
            )
            seed_curve = validation_curve_metrics_df.loc[
                validation_curve_metrics_df["axis"].eq(axis)
                & validation_curve_metrics_df["variant"].isin([candidate, reference])
                & validation_curve_metrics_df["fem_windows"].eq(24_000)
            ].pivot_table(index="seed", columns="variant", values="MAE")
            seeds_better = int((seed_curve[candidate] < seed_curve[reference]).sum())
            p90_allowance = max(
                MATERIAL_ABSOLUTE_ARCSEC,
                MATERIAL_RELATIVE * float(summary.loc[reference, "p90_mean"]),
            )
            bias_allowance = max(
                MATERIAL_ABSOLUTE_ARCSEC,
                MATERIAL_RELATIVE * abs(float(summary.loc[reference, "bias_mean"])),
            )
            checks = {
                "improves_v1_by_required_margin": v2_mae <= v1_mae - threshold,
                "v2_vs_v1_ci95_high_below_zero": float(paired["ci95_high"]) < 0,
                "beats_random_init": v2_mae < float(summary.loc["real_random_init", "MAE_mean"]),
                "not_worse_than_real_SSL": v2_mae <= float(summary.loc["real_SSL", "MAE_mean"]),
                "no_fold_worse_than_0_25": worst_fold_delta <= MAX_SINGLE_FOLD_REGRESSION,
                "p90_not_materially_worse": float(summary.loc[candidate, "p90_mean"]) <= float(summary.loc[reference, "p90_mean"]) + p90_allowance,
                "absolute_bias_not_materially_worse": abs(float(summary.loc[candidate, "bias_mean"])) <= abs(float(summary.loc[reference, "bias_mean"])) + bias_allowance,
                "gain_persists_at_8k_and_24k": persistent_8k_24k,
                "better_in_at_least_two_seeds": seeds_better >= 2,
            }
            gate_rows.append({
                "axis": axis, "priority": priority, "candidate": candidate,
                "reference": reference, "required_MAE_gain": threshold,
                "observed_MAE_gain": v1_mae - v2_mae,
                "v2_vs_v1_ci95_high": float(paired["ci95_high"]),
                "worst_fold_delta_MAE": worst_fold_delta,
                "seeds_better_at_24k": seeds_better, **checks,
                "all_gates_passed": bool(all(checks.values())),
            })
    fem_v2_gate_df = pd.DataFrame(gate_rows)
    display(cv_summary_df)
    display(cv_bootstrap_df)
    display(cv_by_elevation_df)
    display(fem_v2_gate_df)

In [ ]:
locked_test_predictions_df = pd.DataFrame()
locked_test_metrics_df = pd.DataFrame()
locked_test_bootstrap_df = pd.DataFrame()
locked_test_by_elevation_df = pd.DataFrame()
lock_path = OUTPUT_DIR / "LOCKED_TEST_ALREADY_USED.json"

if RUN_LOCKED_TEST:
    if LOCKED_TEST_CONFIRMATION != "I_CONFIRM_GATES_ARE_FROZEN":
        raise RuntimeError('Locked-test confirmation is missing')
    if lock_path.exists():
        raise RuntimeError(f"The locked test has already been used: {lock_path}")
    passing = fem_v2_gate_df.loc[fem_v2_gate_df["all_gates_passed"]].sort_values(
        ["axis", "priority"]
    ).drop_duplicates("axis")
    if passing.empty:
        raise RuntimeError('No axis passes all preregistered gates')

    rows = []
    for decision in passing.itertuples(index=False):
        axis, selected_variant = decision.axis, decision.candidate
        production = production_prediction(axis, train_idx, test_idx)
        append_predictions(
            rows, "locked_test", axis, "production_final", -1, 0, 0,
            test_idx, production,
        )
        seed_predictions = []
        for seed in RANDOM_SEEDS:
            v2_state = run_fem_pretraining(
                fem_train_bank, fem_validation_bank, axis, "v2",
                max(FEM_WINDOW_CURVE), seed,
            )
            _, state, scaler = fit_real_regressor(
                Z_real[train_idx], C_real[train_idx], y_real[axis][train_idx],
                Z_real[val_idx], C_real[val_idx], y_real[axis][val_idx], seed,
                pretrained_state=v2_state,
                reset_supervised_head=selected_variant == "FEM_v2_reset_head",
            )
            model = make_regressor(Z_real.shape[-1], C_real.shape[-1])
            restore_state(model, state)
            seed_predictions.append(scaler.inverse_transform(
                model.predict([Z_real[test_idx], C_real[test_idx]], verbose=0)
            ).reshape(-1))
        candidate = np.mean(seed_predictions, axis=0)
        append_predictions(
            rows, "locked_test", axis, selected_variant, 0, 0,
            max(FEM_WINDOW_CURVE), test_idx, candidate,
        )
    locked_test_predictions_df = pd.DataFrame(rows)
    locked_test_metrics_df = metrics_from_prediction_rows(locked_test_predictions_df)
    bootstrap_rows = []
    for decision in passing.itertuples(index=False):
        axis, selected_variant = decision.axis, decision.candidate
        wide = locked_test_predictions_df.loc[
            locked_test_predictions_df["axis"].eq(axis)
        ].pivot_table(
            index=["row_index", "Timestamp", "target"], columns="variant",
            values="prediction", aggfunc="mean",
        ).reset_index()
        bootstrap_rows.append({
            "axis": axis, "candidate": selected_variant,
            "reference": "production_final",
            **day_block_bootstrap(
                wide["target"], wide[selected_variant],
                wide["production_final"], wide["Timestamp"],
            ),
        })
    locked_test_bootstrap_df = pd.DataFrame(bootstrap_rows)
    locked_test_predictions_df["elevation_band"] = pd.cut(
        locked_test_predictions_df["Elevation"],
        [0, 20, 30, 45, 60, 75, 90], include_lowest=True,
    )
    elevation_rows = []
    for keys, subset in locked_test_predictions_df.groupby(
        ["axis", "variant", "elevation_band"], observed=True
    ):
        elevation_rows.append({
            "axis": keys[0], "variant": keys[1], "elevation_band": keys[2],
            **regression_metrics(subset["target"], subset["prediction"]),
        })
    locked_test_by_elevation_df = pd.DataFrame(elevation_rows)
    lock_path.write_text(json.dumps({
        "axes": passing[["axis", "candidate"]].to_dict(orient="records"),
        "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
        "gate_records": json.loads(fem_v2_gate_df.to_json(orient="records")),
    }, indent=2), encoding="utf-8")
    display(locked_test_metrics_df)
    display(locked_test_bootstrap_df)
    display(locked_test_by_elevation_df)
else:
    print('Final test not run. RUN_LOCKED_TEST=False')

In [ ]:
if len(validation_curve_summary_df):
    plot_data = validation_curve_summary_df.loc[
        validation_curve_summary_df["variant"].str.startswith("FEM")
    ]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)
    for axis_name, ax in zip(["XEL", "EL"], axes):
        subset = plot_data.loc[plot_data["axis"].eq(axis_name)]
        sns.lineplot(
            data=subset, x="fem_windows", y="MAE_mean", hue="variant",
            marker="o", ax=ax,
        )
        ax.set_xscale("log")
        ax.set_title(f"{axis_name}: FEM pretraining sample curve")
        ax.set_xlabel("FEM training windows")
        ax.set_ylabel("Validation MAE [arcsec]")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "fem_v1_v2_sample_curve.png", dpi=200, bbox_inches="tight")
    plt.show()

if len(cv_summary_df):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    for axis_name, ax in zip(["XEL", "EL"], axes):
        subset = cv_summary_df.loc[cv_summary_df["axis"].eq(axis_name)]
        sns.barplot(data=subset, x="variant", y="MAE_mean", color="#0072B2", ax=ax)
        ax.tick_params(axis="x", rotation=30)
        ax.set_title(f"{axis_name}: internal temporal CV")
        ax.set_xlabel("")
        ax.set_ylabel("MAE [arcsec]")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "fem_v2_temporal_cv.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
exports = {
    "real_data_summary.csv": real_data_summary_df,
    "fem_join_audit.csv": fem_join_audit_df,
    "representation_summary.csv": representation_summary_df,
    "fem_window_summary.csv": fem_window_summary_df,
    "fem_window_keys.csv": fem_candidates,
    "ssl_data_summary.csv": ssl_data_summary_df,
    "validation_predictions.csv": validation_predictions_df,
    "validation_curve_metrics.csv": validation_curve_metrics_df,
    "validation_curve_summary.csv": validation_curve_summary_df,
    "validation_v2_vs_v1_bootstrap.csv": validation_v2_vs_v1_bootstrap_df,
    "temporal_cv_predictions.csv": cv_predictions_df,
    "temporal_cv_metrics.csv": cv_metrics_df,
    "temporal_cv_summary.csv": cv_summary_df,
    "temporal_cv_bootstrap.csv": cv_bootstrap_df,
    "temporal_cv_by_elevation.csv": cv_by_elevation_df,
    "fem_v2_gate.csv": fem_v2_gate_df,
    "locked_test_predictions.csv": locked_test_predictions_df,
    "locked_test_metrics.csv": locked_test_metrics_df,
    "locked_test_bootstrap.csv": locked_test_bootstrap_df,
    "locked_test_by_elevation.csv": locked_test_by_elevation_df,
}
for name, frame in exports.items():
    if frame is not None and len(frame):
        frame.to_csv(OUTPUT_DIR / name, index=False)
print('Export completed in:', OUTPUT_DIR)
print('Files written:', sorted(path.name for path in OUTPUT_DIR.glob("*.csv")))

# 16. Output da condividere

Dopo l'esecuzione con `RUN_LOCKED_TEST=False`, condividere:

1. `fem_join_audit_df`
2. `fem_window_summary_df`
3. `validation_curve_summary_df`
4. `validation_v2_vs_v1_bootstrap_df`
5. `cv_summary_df`
6. `cv_bootstrap_df`
7. `cv_by_elevation_df`
8. `fem_v2_gate_df`

Non eseguire il test bloccato finche' `fem_v2_gate_df` non e' stato esaminato
e la configurazione non e' stata congelata.